In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

DRIVE_ROOT = "/content/drive/MyDrive/msc-deepfake"
LOCAL_HF_CACHE = "/content/hf_cache"

os.environ["HF_HOME"] = LOCAL_HF_CACHE
os.environ["HF_HUB_CACHE"] = LOCAL_HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = LOCAL_HF_CACHE
os.makedirs(LOCAL_HF_CACHE, exist_ok=True)

OUT_FOLDER = f"{DRIVE_ROOT}/generated_videos/hunyuan"
METADATA_FOLDER = f"{DRIVE_ROOT}/metadata/hunyuan"
PROMPTS_FILE = f"{DRIVE_ROOT}/batch_prompts/week2_prompts_v1.txt"

os.makedirs(OUT_FOLDER, exist_ok=True)
os.makedirs(METADATA_FOLDER, exist_ok=True)

print(f"OUT_FOLDER: {OUT_FOLDER}")
print(f"METADATA_FOLDER: {METADATA_FOLDER}")

# Confirm A100
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print(f"\nGPU: {result.stdout.strip()}")

OUT_FOLDER: /content/drive/MyDrive/msc-deepfake/generated_videos/hunyuan
METADATA_FOLDER: /content/drive/MyDrive/msc-deepfake/metadata/hunyuan

GPU: NVIDIA A100-SXM4-80GB, 81920 MiB


In [3]:
!pip install -q --upgrade diffusers transformers accelerate imageio imageio-ffmpeg sentencepiece einops safetensors
print("Installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 156.4 MB/s eta 0:00:00
Installed.


In [4]:
from diffusers import HunyuanVideoPipeline, HunyuanVideoTransformer3DModel
import torch, gc

gc.collect()
torch.cuda.empty_cache()

print("Loading HunyuanVideo on A100...")

transformer = HunyuanVideoTransformer3DModel.from_pretrained(
    "hunyuanvideo-community/HunyuanVideo",
    subfolder="transformer",
    torch_dtype=torch.bfloat16
)

pipe = HunyuanVideoPipeline.from_pretrained(
    "hunyuanvideo-community/HunyuanVideo",
    transformer=transformer,
    torch_dtype=torch.bfloat16
)

pipe.to("cuda")
pipe.vae.enable_tiling()
pipe.vae.enable_slicing()

print("HunyuanVideo loaded.")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"GPU total: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading HunyuanVideo on A100...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/510 [00:00<?, ?B/s]

(…)ion_pytorch_model.safetensors.index.json:   0%|          | 0.00/133k [00:00<?, ?B/s]

...

Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

HunyuanVideo loaded.
GPU memory used: 41.43 GB
GPU total: 85.09 GB


In [12]:
import datetime, json
from diffusers.utils import export_to_video

CINEMATIC_SUFFIX = ", natural lighting, cinematic quality, high detail, realistic"

# Full parameter dict — logged alongside every video for reproducibility
GEN_PARAMS = {
    "model_id": "hunyuanvideo-community/HunyuanVideo",
    "height": 480,
    "width": 704,
    "num_frames": 73,
    "num_inference_steps": 20,
    "guidance_scale": 6.0,
    "seed": 42,
    "torch_dtype": "bfloat16",
    "cinematic_suffix": CINEMATIC_SUFFIX,
}

def generate_and_save(prompt_id, category, prompt_text, out_folder, metadata_folder):
    full_prompt = prompt_text + CINEMATIC_SUFFIX
    print(f"\n[{prompt_id}] {category}: {prompt_text[:60]}...")

    start = datetime.datetime.now()
    generator = torch.Generator(device="cuda").manual_seed(GEN_PARAMS["seed"])

    video = pipe(
        prompt=full_prompt,
        height=GEN_PARAMS["height"],
        width=GEN_PARAMS["width"],
        num_frames=GEN_PARAMS["num_frames"],
        num_inference_steps=GEN_PARAMS["num_inference_steps"],
        guidance_scale=GEN_PARAMS["guidance_scale"],
        generator=generator,
    ).frames[0]

    duration = (datetime.datetime.now() - start).total_seconds()
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{prompt_id}_hunyuan_{timestamp}"
    video_path = f"{out_folder}/{filename}.mp4"
    metadata_path = f"{metadata_folder}/{filename}.json"

    export_to_video(video, video_path, fps=24)

    metadata = {
        "video_id": filename,
        "prompt_id": prompt_id,
        "category": category,
        "original_prompt": prompt_text,
        "full_prompt": full_prompt,
        "generator": "HunyuanVideo",
        "generation_time_seconds": duration,
        "generation_datetime": timestamp,
        "gpu": "A100",
        **GEN_PARAMS,
    }
    with open(metadata_path, "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"Saved: {video_path} ({duration:.1f}s = {duration/60:.1f}min)")
    return video_path

print("Function defined with parameter-dict logging.")

Function defined with parameter-dict logging.


In [6]:
with open(PROMPTS_FILE) as f:
    lines = [line.strip() for line in f
             if line.strip() and not line.strip().startswith("#") and "|" in line]

test_line = lines[0]  # w2_001
parts = test_line.split("|", 2)
prompt_id, category, prompt_text = parts

print(f"Sanity check on: {prompt_id}")
generate_and_save(prompt_id, category, prompt_text, OUT_FOLDER, METADATA_FOLDER)

Sanity check on: w2_001

[w2_001] portrait: A woman in her thirties smiling and speaking directly to the...


  0%|          | 0/30 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/hunyuan/w2_001_hunyuan_20260719_132500.mp4 (250.3s = 4.2min)


'/content/drive/MyDrive/msc-deepfake/generated_videos/hunyuan/w2_001_hunyuan_20260719_132500.mp4'

In [8]:
# Find remaining prompts
existing_ids = set()
for v in os.listdir(OUT_FOLDER):
    if v.endswith(".mp4"):
        prompt_id = v.split("_hunyuan_")[0]
        existing_ids.add(prompt_id)

remaining = [line for line in lines if line.split("|")[0] not in existing_ids]

# Take next 5
batch = remaining[:5]
print(f"Already done: {len(existing_ids)}")
print(f"Total remaining: {len(remaining)}")
print(f"This batch: {len(batch)}")
print("=" * 60)

successes = []
failures = []

for line in batch:
    parts = line.split("|", 2)
    if len(parts) != 3:
        continue
    prompt_id, category, prompt_text = parts
    try:
        generate_and_save(prompt_id, category, prompt_text, OUT_FOLDER, METADATA_FOLDER)
        successes.append(prompt_id)
    except Exception as e:
        print(f"FAILED on {prompt_id}: {e}")
        failures.append((prompt_id, str(e)))

# Total progress
now_done = len(existing_ids) + len(successes)
print("\n" + "=" * 60)
print(f"Batch complete. Success: {len(successes)}, Failed: {len(failures)}")
print(f"Total Hunyuan videos now: {now_done}/40")
if failures:
    print(f"Failures: {failures}")

Already done: 1
Total remaining: 39
This batch: 5

[w2_002] portrait: An older man laughing while telling a story, cafe interior, ...


  0%|          | 0/20 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/hunyuan/w2_002_hunyuan_20260719_133044.mp4 (169.5s = 2.8min)

[w2_003] portrait: A young man with glasses reading a book, close-up on his fac...


  0%|          | 0/20 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/hunyuan/w2_003_hunyuan_20260719_133334.mp4 (169.8s = 2.8min)

[w2_004] portrait: A woman with curly hair looking thoughtfully out a rainy win...


...

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/hunyuan/w2_004_hunyuan_20260719_133625.mp4 (170.0s = 2.8min)

[w2_005] portrait: A bearded chef tasting food from a spoon, kitchen background...


  0%|          | 0/20 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/hunyuan/w2_005_hunyuan_20260719_133917.mp4 (171.5s = 2.9min)

[w2_006] hands: Close-up of a person typing on a laptop keyboard, both hands...


  0%|          | 0/20 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/hunyuan/w2_006_hunyuan_20260719_134208.mp4 (171.4s = 2.9min)

Batch complete. Success: 5, Failed: 0
Total Hunyuan videos now: 6/40


In [14]:
from diffusers import HunyuanVideoPipeline, HunyuanVideoTransformer3DModel
import torch, gc

gc.collect()
torch.cuda.empty_cache()

print("Loading HunyuanVideo on A100...")

transformer = HunyuanVideoTransformer3DModel.from_pretrained(
    "hunyuanvideo-community/HunyuanVideo",
    subfolder="transformer",
    torch_dtype=torch.bfloat16
)

pipe = HunyuanVideoPipeline.from_pretrained(
    "hunyuanvideo-community/HunyuanVideo",
    transformer=transformer,
    torch_dtype=torch.bfloat16
)

pipe.to("cuda")
pipe.vae.enable_tiling()
pipe.vae.enable_slicing()

print("HunyuanVideo loaded (clean, no compile).")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Loading HunyuanVideo on A100...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

HunyuanVideo loaded (clean, no compile).
GPU memory used: 67.08 GB


In [15]:
import datetime, json
from diffusers.utils import export_to_video

CINEMATIC_SUFFIX = ", natural lighting, cinematic quality, high detail, realistic"

# Full parameter dict — logged alongside every video for reproducibility
GEN_PARAMS = {
    "model_id": "hunyuanvideo-community/HunyuanVideo",
    "height": 480,
    "width": 704,
    "num_frames": 73,
    "num_inference_steps": 20,
    "guidance_scale": 6.0,
    "seed": 42,
    "torch_dtype": "bfloat16",
    "cinematic_suffix": CINEMATIC_SUFFIX,
}

def generate_and_save(prompt_id, category, prompt_text, out_folder, metadata_folder):
    full_prompt = prompt_text + CINEMATIC_SUFFIX
    print(f"\n[{prompt_id}] {category}: {prompt_text[:60]}...")

    start = datetime.datetime.now()
    generator = torch.Generator(device="cuda").manual_seed(GEN_PARAMS["seed"])

    video = pipe(
        prompt=full_prompt,
        height=GEN_PARAMS["height"],
        width=GEN_PARAMS["width"],
        num_frames=GEN_PARAMS["num_frames"],
        num_inference_steps=GEN_PARAMS["num_inference_steps"],
        guidance_scale=GEN_PARAMS["guidance_scale"],
        generator=generator,
    ).frames[0]

    duration = (datetime.datetime.now() - start).total_seconds()
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{prompt_id}_hunyuan_{timestamp}"
    video_path = f"{out_folder}/{filename}.mp4"
    metadata_path = f"{metadata_folder}/{filename}.json"

    export_to_video(video, video_path, fps=24)

    metadata = {
        "video_id": filename,
        "prompt_id": prompt_id,
        "category": category,
        "original_prompt": prompt_text,
        "full_prompt": full_prompt,
        "generator": "HunyuanVideo",
        "generation_time_seconds": duration,
        "generation_datetime": timestamp,
        "gpu": "A100",
        **GEN_PARAMS,
    }
    with open(metadata_path, "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"Saved: {video_path} ({duration:.1f}s = {duration/60:.1f}min)")
    return video_path

print("Function defined with parameter-dict logging.")

Function defined with parameter-dict logging.


In [16]:
# Find remaining prompts
existing_ids = set()
for v in os.listdir(OUT_FOLDER):
    if v.endswith(".mp4"):
        prompt_id = v.split("_hunyuan_")[0]
        existing_ids.add(prompt_id)

remaining = [line for line in lines if line.split("|")[0] not in existing_ids]

# Take next 5
batch = remaining[:5]
print(f"Already done: {len(existing_ids)}")
print(f"Total remaining: {len(remaining)}")
print(f"This batch: {len(batch)}")
print("=" * 60)

successes = []
failures = []

for line in batch:
    parts = line.split("|", 2)
    if len(parts) != 3:
        continue
    prompt_id, category, prompt_text = parts
    try:
        generate_and_save(prompt_id, category, prompt_text, OUT_FOLDER, METADATA_FOLDER)
        successes.append(prompt_id)
    except Exception as e:
        print(f"FAILED on {prompt_id}: {e}")
        failures.append((prompt_id, str(e)))

# Total progress
now_done = len(existing_ids) + len(successes)
print("\n" + "=" * 60)
print(f"Batch complete. Success: {len(successes)}, Failed: {len(failures)}")
print(f"Total Hunyuan videos now: {now_done}/40")
if failures:
    print(f"Failures: {failures}")

Already done: 6
Total remaining: 34
This batch: 5

[w2_007] hands: A hand pouring milk from a glass bottle into a coffee cup, k...


  0%|          | 0/20 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/hunyuan/w2_007_hunyuan_20260719_134948.mp4 (171.2s = 2.9min)

[w2_008] hands: Someone tying their shoelaces, close-up on hands and shoes, ...


  0%|          | 0/20 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/hunyuan/w2_008_hunyuan_20260719_135240.mp4 (171.5s = 2.9min)

[w2_009] hands: A person writing in a notebook with a fountain pen, close-up...


...

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/hunyuan/w2_009_hunyuan_20260719_135532.mp4 (171.4s = 2.9min)

[w2_010] hands: Two hands shuffling a deck of playing cards on a wooden tabl...


  0%|          | 0/20 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/hunyuan/w2_010_hunyuan_20260719_135824.mp4 (171.5s = 2.9min)

[w2_011] multi_person: Two friends walking down a busy street talking to each other...


  0%|          | 0/20 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/hunyuan/w2_011_hunyuan_20260719_140117.mp4 (171.4s = 2.9min)

Batch complete. Success: 5, Failed: 0
Total Hunyuan videos now: 11/40


In [17]:
# Find remaining prompts
existing_ids = set()
for v in os.listdir(OUT_FOLDER):
    if v.endswith(".mp4"):
        prompt_id = v.split("_hunyuan_")[0]
        existing_ids.add(prompt_id)

remaining = [line for line in lines if line.split("|")[0] not in existing_ids]

# Take next 10
batch = remaining[:10]
print(f"Already done: {len(existing_ids)}")
print(f"Total remaining: {len(remaining)}")
print(f"This batch: {len(batch)}")
print("=" * 60)

successes = []
failures = []

for line in batch:
    parts = line.split("|", 2)
    if len(parts) != 3:
        continue
    prompt_id, category, prompt_text = parts
    try:
        generate_and_save(prompt_id, category, prompt_text, OUT_FOLDER, METADATA_FOLDER)
        successes.append(prompt_id)
    except Exception as e:
        print(f"FAILED on {prompt_id}: {e}")
        failures.append((prompt_id, str(e)))

# Total progress
now_done = len(existing_ids) + len(successes)
print("\n" + "=" * 60)
print(f"Batch complete. Success: {len(successes)}, Failed: {len(failures)}")
print(f"Total Hunyuan videos now: {now_done}/40")
if failures:
    print(f"Failures: {failures}")

Already done: 11
Total remaining: 29
This batch: 10

[w2_012] multi_person: A family of four eating dinner at a dining table, warm eveni...


  0%|          | 0/20 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/hunyuan/w2_012_hunyuan_20260719_140415.mp4 (171.3s = 2.9min)

[w2_013] multi_person: Three colleagues in a meeting room having a discussion, whit...


  0%|          | 0/20 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/hunyuan/w2_013_hunyuan_20260719_140707.mp4 (171.4s = 2.9min)

[w2_014] multi_person: A parent teaching a child to ride a bicycle in a park, sunny...


...

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/hunyuan/w2_019_hunyuan_20260719_142419.mp4 (170.9s = 2.8min)

[w2_020] motion: A tennis player serving a ball on a clay court, dust visible...


  0%|          | 0/20 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/hunyuan/w2_020_hunyuan_20260719_142711.mp4 (171.3s = 2.9min)

[w2_021] text_scene: A shopkeeper standing in front of a bookstore, storefront si...


  0%|          | 0/20 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/hunyuan/w2_021_hunyuan_20260719_143003.mp4 (171.2s = 2.9min)

Batch complete. Success: 10, Failed: 0
Total Hunyuan videos now: 21/40


In [18]:
# Find remaining prompts
existing_ids = set()
for v in os.listdir(OUT_FOLDER):
    if v.endswith(".mp4"):
        prompt_id = v.split("_hunyuan_")[0]
        existing_ids.add(prompt_id)

remaining = [line for line in lines if line.split("|")[0] not in existing_ids]

# Take remaining
batch = remaining[:19]
print(f"Already done: {len(existing_ids)}")
print(f"Total remaining: {len(remaining)}")
print(f"This batch: {len(batch)}")
print("=" * 60)

successes = []
failures = []

for line in batch:
    parts = line.split("|", 2)
    if len(parts) != 3:
        continue
    prompt_id, category, prompt_text = parts
    try:
        generate_and_save(prompt_id, category, prompt_text, OUT_FOLDER, METADATA_FOLDER)
        successes.append(prompt_id)
    except Exception as e:
        print(f"FAILED on {prompt_id}: {e}")
        failures.append((prompt_id, str(e)))

# Total progress
now_done = len(existing_ids) + len(successes)
print("\n" + "=" * 60)
print(f"Batch complete. Success: {len(successes)}, Failed: {len(failures)}")
print(f"Total Hunyuan videos now: {now_done}/40")
if failures:
    print(f"Failures: {failures}")

Already done: 21
Total remaining: 19
This batch: 19

[w2_022] text_scene: A newspaper vendor at a street corner with newspapers displa...


  0%|          | 0/20 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/hunyuan/w2_022_hunyuan_20260719_143310.mp4 (171.1s = 2.9min)

[w2_023] text_scene: A person walking past a chalkboard menu outside a cafe...


  0%|          | 0/20 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/hunyuan/w2_023_hunyuan_20260719_143601.mp4 (171.3s = 2.9min)

[w2_024] text_scene: A traveller checking a departure board at a train station...


...

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/hunyuan/w2_038_hunyuan_20260719_151858.mp4 (171.3s = 2.9min)

[w2_039] edge_case: A crowded market at night with many people walking, string l...


  0%|          | 0/20 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/hunyuan/w2_039_hunyuan_20260719_152150.mp4 (171.3s = 2.9min)

[w2_040] edge_case: A person walking their dog past a shop window, both dog and ...


  0%|          | 0/20 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/hunyuan/w2_040_hunyuan_20260719_152442.mp4 (171.3s = 2.9min)

Batch complete. Success: 19, Failed: 0
Total Hunyuan videos now: 40/40
